# 04. 방법 2: 선형 프로브

이미지 인코더를 고정하고 그 임베딩으로 로지스틱 회귀만 학습한다. 런타임: GPU.

1. 고정된 이미지 인코더로 train/val/test 임베딩을 추출한다 (증강 없음).
2. 로지스틱 회귀를 학습하고, 검증 세트 정확도로 정규화 강도 C를 고른다.

In [ ]:
# 공통 셋업: 저장소 clone, 패키지 설치, Drive 연결
import os, sys
REPO_URL = "https://github.com/Cranezero/aix-assignment-pr.git"
REPO_DIR = "/content/aix-assignment-pr"
if not os.path.exists(REPO_DIR):
    !git clone -q {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull -q
%cd {REPO_DIR}
!pip install -q -r requirements.txt
sys.path.insert(0, REPO_DIR)

from google.colab import drive
drive.mount("/content/drive")

In [ ]:
import time
import numpy as np
import pandas as pd
import torch
from sklearn.linear_model import LogisticRegression
from torch.utils.data import DataLoader

from src import config
from src.clip_utils import encode_images, load_clip
from src.data import StreetDataset, get_transform, load_split, prepare_images
from src.metrics import save_results
from src.utils import export_outputs, get_device, restore_outputs, set_seed

restore_outputs()  # results/, assets/ 폴더 생성 + Drive에 저장된 이전 결과 가져오기

set_seed(config.SEED)
device = get_device()
image_dir = prepare_images()
model, _ = load_clip(device)

In [ ]:
splits = {s: load_split(s) for s in ["train", "val", "test"]}
emb, lab, enc_sec = {}, {}, {}
for s, df in splits.items():
    loader = DataLoader(StreetDataset(df, image_dir, get_transform(train=False)),
                        batch_size=256, shuffle=False, num_workers=2)
    t0 = time.time()
    e, y = encode_images(model, loader, device)
    enc_sec[s] = time.time() - t0
    emb[s], lab[s] = e.numpy(), y.numpy()
    print(s, emb[s].shape, f"{enc_sec[s]:.1f}s")

# 임베딩은 Drive에 저장해 두고 재사용할 수 있게 한다
torch.save({"emb": emb, "lab": lab, "ids": {s: df["id"].tolist() for s, df in splits.items()}},
           config.DRIVE_DIR / "clip_embeddings.pt")

## 정규화 강도 C 선택

In [ ]:
search = []
for C in np.logspace(-2, 3, 11):
    t0 = time.time()
    clf = LogisticRegression(C=C, max_iter=3000, random_state=config.SEED)
    clf.fit(emb["train"], lab["train"])
    search.append({"C": C, "train_acc": clf.score(emb["train"], lab["train"]),
                   "val_acc": clf.score(emb["val"], lab["val"]), "fit_sec": time.time() - t0})
search = pd.DataFrame(search)
search.to_csv(config.RESULTS_DIR / "linear_probe_c_search.csv", index=False)
display(search.round(4))
best_C = float(search.loc[search["val_acc"].idxmax(), "C"])
print("best C:", best_C)

In [ ]:
t0 = time.time()
clf = LogisticRegression(C=best_C, max_iter=3000, random_state=config.SEED)
clf.fit(emb["train"], lab["train"])
fit_sec = time.time() - t0

probs = clf.predict_proba(emb["test"])
metrics = save_results("linear_probe", splits["test"]["id"], probs, lab["test"],
                       extra={"best_C": best_C, "val_acc": clf.score(emb["val"], lab["val"]),
                              # 학습 시간 = train/val 임베딩 추출 + 선택된 C로 학습
                              "train_time_sec": enc_sec["train"] + enc_sec["val"] + fit_sec,
                              "fit_sec": fit_sec, "c_search_sec": float(search["fit_sec"].sum())})
print({k: round(v, 4) for k, v in metrics.items() if k in ("top1", "top5", "macro", "train_time_sec")})
per_country = pd.Series(metrics["per_country"]).rename("acc").to_frame()
per_country.insert(0, "name", [config.COUNTRIES[c] for c in per_country.index])
display(per_country.sort_values("acc", ascending=False).round(3))

In [ ]:
export_outputs()